In [2]:
from langchain_core.tools import tool

@tool
def division(a:int,b:int)->int:
    """divide the number"""
    return a/b
    

In [ ]:
from typing import Annotated,List

@tool
async def multiply_by_max(
    a: Annotated[int,"A value"],
    b: Annotated[List[int], "list of int over which to take maximum"]
)->int:
    """multiply a by the maximum of b """
    return a*max(b)

In [6]:
print(multiply_by_max.args)

{'a': {'description': 'A value', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'list of int over which to take maximum', 'items': {'type': 'integer'}, 'title': 'B', 'type': 'array'}}


In [9]:
##structure tool

from langchain_core.tools import StructuredTool


def mutlitply(a:int,b:int)->int:
    """divide the number"""
    return a*b

async def amutlitply(a:int,b:int)->int:
    """divide the number"""
    return a*b

calculator = StructuredTool.from_function(func=mutlitply,coroutine=amutlitply)

print(calculator.invoke({"a":2,"b":3}))
print(await calculator.ainvoke({"a":2,"b":3}))
    


6
6


In [11]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper=WikipediaAPIWrapper(top_k_results=5,doc_content_chars_max=500)
tool=WikipediaQueryRun(api_wrapper=api_wrapper)

print(tool.invoke({"query":"langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.



Page: Vector database
Summary: A vector database, vector store or vector search engine is a database that uses the vector space model to store vectors


In [12]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=100)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

print(wiki_tool.invoke({"query": "langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of 


In [14]:
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int:
    """Adds a and b."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiplies a and b."""
    return a * b

In [16]:
tools = [wiki_tool,add,multiply]

In [15]:
from langchain.chat_models import init_chat_model
llm=init_chat_model("qwen/qwen3-32b",model_provider="groq")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000000CFFEEB5160>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000000CFFEEB5E80>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [17]:
llm_with_tools=llm.bind_tools(tools)
llm_with_tools

RunnableBinding(bound=ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000000CFFEEB5160>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000000CFFEEB5E80>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'wikipedia', 'description': 'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.', 'parameters': {'properties': {'query': {'description': 'query to look up on wikipedia', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'add', 'description': 'Adds a and b.', 'parameters': {'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'multiply', 'descripti

In [18]:
from langchain_core.messages import HumanMessage
query="What is 2* 3"
messages = [HumanMessage(query)]
response=llm_with_tools.invoke(query)
print(response)

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking "What is 2* 3". Let me see. The tools available include add and multiply functions. The question is a multiplication, so I should use the multiply function. The parameters are a and b, both integers. Here, a is 2 and b is 3. So I need to call the multiply function with these values. Let me double-check the function\'s parameters to make sure. Yep, multiply takes two integers. So the correct tool call is multiply with arguments 2 and 3.\n', 'tool_calls': [{'id': 'd6jcpgaqm', 'function': {'arguments': '{"a":2,"b":3}', 'name': 'multiply'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 139, 'prompt_tokens': 325, 'total_tokens': 464, 'completion_time': 0.232751159, 'prompt_time': 0.014675842, 'queue_time': 0.239492118, 'total_time': 0.247427001}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_efa9879028', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprob

In [19]:

response.tool_calls

[{'name': 'multiply',
  'args': {'a': 2, 'b': 3},
  'id': 'd6jcpgaqm',
  'type': 'tool_call'}]

In [22]:
for tool_call in response.tool_calls:
    selected_tool = {"add": add, "multiply": multiply,"wikipedia":wiki_tool}[tool_call["name"].lower()]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    
messages

[HumanMessage(content='What is 2* 3', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='6', name='multiply', tool_call_id='d6jcpgaqm')]

In [23]:
llm_with_tools.invoke(messages)

AIMessage(content='The result of multiplying 2 by 3 is 6.\n\nThe answer is 6.', additional_kwargs={'reasoning_content': 'Okay, the user asked "What is 2* 3". Let me see. The tools available include the multiply function. The multiply function takes two integers, a and b. So 2 multiplied by 3 would be 6. The answer is 6.\n'}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 335, 'total_tokens': 414, 'completion_time': 0.166632651, 'prompt_time': 0.016048643, 'queue_time': 0.051505476, 'total_time': 0.182681294}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--471ba7c3-65c7-4fc8-85a9-161196cee56f-0', usage_metadata={'input_tokens': 335, 'output_tokens': 79, 'total_tokens': 414})

In [24]:
query="What is langchain and what is 5*15"
messages = [HumanMessage(query)]
response=llm_with_tools.invoke(query)
print(response)

content='' additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user is asking two things here: what is LangChain and what is 5 multiplied by 15. First, I need to figure out how to handle both parts.\n\nFor the first part about LangChain, I should use the Wikipedia tool to get an accurate definition. The user probably wants a general explanation, so the Wikipedia function is the right choice here. The query should be "LangChain" to look up the term.\n\nThen, the second part is a math problem: 5 times 15. Multiplying these numbers requires using the multiply function. The parameters are a=5 and b=15. Since there\'s a direct function for multiplication, that\'s straightforward.\n\nWait, but do I need to call both functions? Yes, because the user is asking two separate questions. I\'ll need to make two tool calls. First, handle the Wikipedia lookup for LangChain, then perform the multiplication. Let me structure the tool calls properly in JSON format as specified. Make sure the 

In [25]:
for tool_call in response.tool_calls:
    selected_tool = {"add": add, "multiply": multiply,"wikipedia":wiki_tool}[tool_call["name"].lower()]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    
messages

[HumanMessage(content='What is langchain and what is 5*15', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='Page: LangChain\nSummary: LangChain is a software framework that helps facilitate the integration of ', name='wikipedia', tool_call_id='0jp5ngpq1'),
 ToolMessage(content='75', name='multiply', tool_call_id='wdk15477v')]

In [26]:
llm_with_tools.invoke(messages)

AIMessage(content="LangChain is a software framework designed to simplify the integration of large language models (LLMs) into applications. It provides tools for tasks like prompting, data retrieval, and workflow orchestration, enabling developers to build AI-driven applications more efficiently. \n\nAs for the math question:  \n**5 × 15 = 75**.  \n\nLet me know if you'd like further details! 😊", additional_kwargs={'reasoning_content': "Okay, let's tackle the user's question. They asked two things: what LangChain is and what 5 multiplied by 15 is. \n\nFirst, for the LangChain part. The user probably wants a clear, concise definition. I remember that LangChain is a framework for building applications with large language models. The Wikipedia tool was already used, and the summary mentions it's a software framework that integrates with LLMs. I should present that information in a friendly way, maybe expanding a bit on how it helps in creating apps.\n\nNext, the math part: 5 times 15. Th